<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/ML_Examples/05_FuncApprox_2D_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2D Function Approximation using PyTorch

Train a multi-layer perceptron (MLP) neural network to approximate a 2D function:

$$f(x_1, x_2) = \sin\left(\frac{\pi}{2} x_1\right) \cos\left(\frac{\pi}{4} x_2\right)$$

where $x_1, x_2 \in [-2, 2]$.

---

**ELC 5365: Deep Learning** | Baylor University

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Define the Target Function

We define the 2D function that we want our neural network to learn:

$$f(x_1, x_2) = \sin\left(\frac{\pi}{2} x_1\right) \cos\left(\frac{\pi}{4} x_2\right)$$

In [ ]:
def target_function(x1, x2):
    """The 2D function to approximate."""
    return np.sin(np.pi * x1 / 2.0) * np.cos(np.pi * x2 / 4.0)

## 3. Visualize the Original Function

In [ ]:
# Create a meshgrid for visualization
A = 2  # Domain: [-A, A]
grid_size = 100
x1_vals = np.linspace(-A, A, grid_size)
x2_vals = np.linspace(-A, A, grid_size)
X1, X2 = np.meshgrid(x1_vals, x2_vals)
Z_original = target_function(X1, X2)

# Plot contour and surface
fig = plt.figure(figsize=(14, 5))
fig.suptitle('Original Function: $f(x_1, x_2) = \sin(\pi x_1/2) \cos(\pi x_2/4)$', fontsize=14)

# Contour plot
ax1 = fig.add_subplot(1, 2, 1)
cont = ax1.contourf(X1, X2, Z_original, levels=30, cmap='viridis')
fig.colorbar(cont, ax=ax1)
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_title('Contour Plot')

# 3D surface plot
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1, X2, Z_original, cmap='viridis', edgecolor='none', alpha=0.9)
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$f(x_1, x_2)$')
ax2.set_title('Surface Plot')

plt.tight_layout()
plt.show()

## 4. Generate Training Data

We randomly sample points from the domain $[-2, 2] \times [-2, 2]$ and compute the corresponding function values.

In [ ]:
# Generate training data
N = 5000  # Number of training samples
x1_train = np.random.uniform(-A, A, N)
x2_train = np.random.uniform(-A, A, N)
y_train = target_function(x1_train, x2_train)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(np.column_stack([x1_train, x2_train]))
Y_train = torch.FloatTensor(y_train).unsqueeze(1)  # shape: (N, 1)

print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")

## 5. Define the Neural Network

We use a multi-layer perceptron with two hidden layers, each containing 200 neurons with ReLU activation:

$$\text{Input (2)} \rightarrow \text{Hidden (200, ReLU)} \rightarrow \text{Hidden (200, ReLU)} \rightarrow \text{Output (1)}$$

In [ ]:
class FuncApproxNet(nn.Module):
    def __init__(self, n_hidden=200):
        super(FuncApproxNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1)
        )

    def forward(self, x):
        return self.net(x)

model = FuncApproxNet(n_hidden=200)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

## 6. Train the Model

We use Mean Squared Error (MSE) loss and the Adam optimizer.

In [ ]:
# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 2000
batch_size = 256
loss_history = []

for epoch in range(num_epochs):
    # Mini-batch training
    perm = torch.randperm(N)
    epoch_loss = 0.0
    num_batches = 0

    for i in range(0, N, batch_size):
        idx = perm[i:i+batch_size]
        X_batch = X_train[idx]
        Y_batch = Y_train[idx]

        # Forward pass
        y_pred = model(X_batch)
        loss = criterion(y_pred, Y_batch)

        # Backward pass and update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    loss_history.append(avg_loss)

    if (epoch + 1) % 200 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}')

## 7. Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Over Epochs')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## 8. Evaluate and Visualize the Approximation

We compare the original function with the neural network's approximation using contour and 3D surface plots.

In [ ]:
# Generate evaluation grid
x1_eval = np.linspace(-A, A, grid_size)
x2_eval = np.linspace(-A, A, grid_size)
X1_eval, X2_eval = np.meshgrid(x1_eval, x2_eval)

# Original function values
Z_original = target_function(X1_eval, X2_eval)

# Neural network predictions
X_test = torch.FloatTensor(
    np.column_stack([X1_eval.flatten(), X2_eval.flatten()]))
with torch.no_grad():
    Z_predicted = model(X_test).numpy().reshape(grid_size, grid_size)

# Compute approximation error
Z_error = np.abs(Z_original - Z_predicted)
print(f"Max absolute error: {Z_error.max():.6f}")
print(f"Mean absolute error: {Z_error.mean():.6f}")

In [ ]:
# --- Contour Plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original
c1 = axes[0].contourf(X1_eval, X2_eval, Z_original, levels=30, cmap='viridis')
fig.colorbar(c1, ax=axes[0])
axes[0].set_title('Original Function')
axes[0].set_xlabel('$x_1$')
axes[0].set_ylabel('$x_2$')

# Approximation
c2 = axes[1].contourf(X1_eval, X2_eval, Z_predicted, levels=30, cmap='viridis')
fig.colorbar(c2, ax=axes[1])
axes[1].set_title('Neural Network Approximation')
axes[1].set_xlabel('$x_1$')
axes[1].set_ylabel('$x_2$')

# Error
c3 = axes[2].contourf(X1_eval, X2_eval, Z_error, levels=30, cmap='hot')
fig.colorbar(c3, ax=axes[2])
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('$x_1$')
axes[2].set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

In [ ]:
# --- 3D Surface Plots ---
fig = plt.figure(figsize=(16, 6))

# Original function surface
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X1_eval, X2_eval, Z_original,
                 cmap='viridis', edgecolor='none', alpha=0.9)
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_zlabel('$f(x_1, x_2)$')
ax1.set_title('Original Function')

# Neural network approximation surface
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1_eval, X2_eval, Z_predicted,
                 cmap='viridis', edgecolor='none', alpha=0.9)
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$\hat{f}(x_1, x_2)$')
ax2.set_title('Neural Network Approximation')

plt.tight_layout()
plt.show()

## 9. Experiment: Approximate a Different Function

Now try approximating a more complex function:

$$f_2(x_1, x_2) = e^{\sin\left(\sqrt{x_1^2 + x_2^2}\right)}, \quad x_1, x_2 \in [-10, 10]$$

In [ ]:
def target_function_2(x1, x2):
    """A more complex 2D function to approximate."""
    return np.exp(np.sin(np.sqrt(x1**2 + x2**2)))

# Generate training data for f2
A2 = 10
N2 = 10000
x1_train2 = np.random.uniform(-A2, A2, N2)
x2_train2 = np.random.uniform(-A2, A2, N2)
y_train2 = target_function_2(x1_train2, x2_train2)

X_train2 = torch.FloatTensor(np.column_stack([x1_train2, x2_train2]))
Y_train2 = torch.FloatTensor(y_train2).unsqueeze(1)

# Create and train model
model2 = FuncApproxNet(n_hidden=200)
criterion2 = nn.MSELoss()
optimizer2 = optim.Adam(model2.parameters(), lr=0.001)

num_epochs2 = 2000
batch_size2 = 512

for epoch in range(num_epochs2):
    perm = torch.randperm(N2)
    for i in range(0, N2, batch_size2):
        idx = perm[i:i+batch_size2]
        y_pred = model2(X_train2[idx])
        loss = criterion2(y_pred, Y_train2[idx])
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()

    if (epoch + 1) % 400 == 0:
        with torch.no_grad():
            total_loss = criterion2(model2(X_train2), Y_train2)
        print(f'Epoch [{epoch+1}/{num_epochs2}], Loss: {total_loss.item():.6f}')

In [ ]:
# Visualize f2 results
x1_e = np.linspace(-A2, A2, 100)
x2_e = np.linspace(-A2, A2, 100)
X1_e, X2_e = np.meshgrid(x1_e, x2_e)

Z_orig2 = target_function_2(X1_e, X2_e)

X_t2 = torch.FloatTensor(np.column_stack([X1_e.flatten(), X2_e.flatten()]))
with torch.no_grad():
    Z_pred2 = model2(X_t2).numpy().reshape(100, 100)

fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X1_e, X2_e, Z_orig2, cmap='viridis', edgecolor='none')
ax1.set_title('Original: $f_2(x_1, x_2) = e^{\sin(\sqrt{x_1^2 + x_2^2})}$')
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1_e, X2_e, Z_pred2, cmap='viridis', edgecolor='none')
ax2.set_title('Neural Network Approximation')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

error2 = np.abs(Z_orig2 - Z_pred2)
print(f"Max absolute error: {error2.max():.6f}")
print(f"Mean absolute error: {error2.mean():.6f}")

## 10. Exercises

1. **Change the number of hidden neurons**: Try `n_hidden = 50` and `n_hidden = 400`. How does it affect the approximation quality?

2. **Add more hidden layers**: Modify `FuncApproxNet` to have 3 or 4 hidden layers. Does deeper always mean better?

3. **Try different activation functions**: Replace `nn.ReLU()` with `nn.Tanh()` or `nn.LeakyReLU()`. Compare results.

4. **Change the learning rate**: Try `lr = 0.01` and `lr = 0.0001`. How does it affect training speed and final accuracy?

5. **Define your own function**: Create a new target function and train the network to approximate it.